In [ ]:
最適なペア順に信号線値を並べ替えたファイル（s~simular_output）をもとに、最適なペアの信号線を統合し、正解データを作成
※s~simular_outputファイルは、s~outputファイルの行と列が入れ替わっていることに注意⇒詳しくはokinaka.txtを参照

In [ ]:
2025/05/26

# 学習済みのANNを用いて学習を行うプログラム
# cd workspace/research2/experiment
#　実行コマンド：　python3 diagnosis.py

import numpy as np
import logging
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential, model_from_json
from keras.layers import Dense, Dropout, BatchNormalization, Activation
from keras.layers import LeakyReLU
from tensorflow.keras.utils import get_custom_objects
from keras import optimizers
from keras import backend as K
from tensorflow.keras.optimizers import Adam

import matplotlib
import matplotlib.pyplot as plt
from memory_profiler import memory_usage
import time
import pathlib
import sklearn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix

import sys
import argparse
import os
import multiprocessing
from multiprocessing import Process

import random

#グローバル変数
cir = 's344'  # 対象回路
tp_file = cir + '.vec'  # テストパターンファイル名
part_stdic_file = cir + "stdic_bi/aout_" #縮退故障辞書ファイル名の一部
part_brdic_file = cir + "brdic_bi/aout_" #ブリッジ故障辞書ファイルの一部
diagnosis_dir = cir + "diagnosis_data/" #故障診断を行うためのデータを保存するフォルダ
target_fault_num = 30  # 故障診断対象の故障の数⇒全ての故障の中からtarget_fault_num個をランダムに選択する
correct_output_file = 'correct_output/' + cir + '_correct_output'  # 正常な回路の出力ファイル
input_data_file = cir + 'input'  # 入力データファイル
correct_data_file = cir + '分割正解データ' + '/' + cir + 'integrated_output'  # 統合後の正解データファイル
suplit_num_file = cir + '分割正解データ' + '/' + cir + 'suplit_num'  # モデルの分割数が保存されたファイル
suplit_data_num_file = cir + '分割正解データ' + '/' + cir + 'suplit_data_num'  # データの分割数が保存されたファイル
input_data_num = None  # 1個のモデルにおける学習データ数
input_node_num = None  #入力層におけるノード数 初期値は8　学習結果によって変更
output_node_num = None #　出力層のノード数＝分割数による
num_models = None  # 学習させるモデルの数
model_folder = cir + 'sepmodel'  # 学習済みモデルを保存するフォルダ
fault_line_num = None  # 故障診断対象の回路における故障信号線の総数
fault_type_sum = 12  # 故障の種類の総数

processes = 8  # 並列処理のプロセス数

def mk_input_data():
    # 入力データファイルを開いてデータを読み込む
    with open(input_data_file) as f:
        lines = [_.replace(",", "").replace("\n", "") for _ in f.readlines()]

    # print(lines)
    
    global input_data_num
    input_data_num = int(len(lines))      #学習データ数を設定。学習データ数は入力データの行数
    global input_node_num               #グローバル変数を書き換え
    input_node_num = int(len(lines[0]))      #入力ノード数を設定。入力ノード数は入力データの各行の要素数

    int_lines = [list(map(int, _)) for _ in lines]  #list要素の型をint型に変換

    return np.array(int_lines)


def mk_output_data(fname):
    # 正解データファイルを開いてデータを読み込む
    with open(fname) as f:
        lines = [_.replace("\n", "") for _ in f.readlines()]
    
    lines = [value.split(",") for value in lines] # カンマ区切りで各信号線をリストに格納
    
    # print("dsadsa")
    # print(lines[0])

    for i in range(len(lines)):
        for j in range(len(lines[i])):
            lines[i][j] = float(lines[i][j])
    
    # print("gaga")
    # print(lines[0])
    
    global output_node_num               #グローバル変数を書き換え
    output_node_num = int(len(lines[0]))      #出力ノード数を設定。出力ノード数は正解データの各行の要素数


if __name__ == '__main__':
    
    # テストパターン数を取得
    with open(tp_file, 'r') as f:
        line = f.readline()
        tp_num = int(line.split()[0])  # テストパターン数
        print(tp_num)

    # 故障診断対象の回路における故障の総数を取得
    part_stdic_file0 =  part_stdic_file + str(0)  # 故障辞書ファイルの1つ
    with open(part_stdic_file0, 'r') as f:
        fault_inf = f.readline()  # 故障情報
        fault_num = int(fault_inf.split()[2])  # 対象回路で起こりうる故障数
        lines = f.readlines()[::2]  # 故障辞書ファイルの全行を読み込む
        st_fault_all_line = [int(line.split()[3]) for line in lines[::2]]  # 故障信号線のIDを取得
        print(fault_num)
        print("st_fault_all_line：", st_fault_all_line)
        # print(suplit_num)

    # 縮退故障が発生する信号線をtarget_fault_num個ランダムに選択。さらに、その信号線で発生するのは0縮退故障か1縮退故障かをランダムに決定する
    st_fault_target_line = random.sample(st_fault_all_line, target_fault_num) # 想定する故障信号線をランダムに選ぶ
    st_fault_target_line = sorted(st_fault_target_line) # 小さい番号順に並び替える
    st_fault_target_type = random.choices([0, 1], k=target_fault_num)  # 0縮退故障か1縮退故障かをランダムに決定する
    with open(diagnosis_dir + 'st_fault_line', 'w') as f: # 故障IDをファイルに保存
        f.write('診断対象縮退故障数' + str(target_fault_num) + '\n')  # 故障の数をファイルに保存
        for i in range(target_fault_num):
            f.write(str(st_fault_target_line[i]) + ' ' + "sa " + str(st_fault_target_type[i]) + "\n")

    print(st_fault_target_line)
    print(st_fault_target_type)

    # 診断対象故障の出力値をファイルに保存
    for i in range(tp_num):
      stdic_file = part_stdic_file + str(i)  # 故障辞書ファイルを指定
      with open(stdic_file, 'r') as f:
        fault_inf = f.readline()  # 故障情報
        lines = f.readlines()  # 故障辞書ファイルの全行を読み込む

      # 各辞書におけるIDと出力値を取得
      fault_output = [0 for _ in range(target_fault_num)]  # 故障出力値を格納するリスト
      idx = 0
      for j in range(0, len(lines), 2):
        if st_fault_target_line[idx] == int(lines[j].split()[3]) and st_fault_target_type[idx] == int(lines[j].split()[5]): # ファイルの「id 0 Fault 1 sa 0」の行から、信号線番号と0、1縮退故障情報を取得して、比較
            fault_output[idx] = lines[j+1].replace('\n', '')  # 故障出力値を取得
            idx += 1
        if idx == target_fault_num:  # 診断対象の故障数に達したら、ループを抜ける。これは、診断対象の故障信号線が小さい順に並び替えられているため、target_fault_num個の故障を取得したら、ループを抜ける
            break

      # 取得した故障出力値をファイルに保存
      with open(diagnosis_dir + 'fault_output' + str(i), 'w') as f:
          for j in range(len(fault_output)):
              f.write(fault_output[j] + '\n')
    
    print(f"fault_output：{fault_output}")
    
    #故障出力を正常な回路出力と比較して、パスフェイル情報を取得する
    correct_output_value = []  # 正常な回路出力を格納するリスト
    with open(correct_output_file, 'r') as f:
        lines = f.readlines()  # 正常な回路出力の全行を読み込む
        for i in range(1, len(lines), 2):
            correct_output_value.append(lines[i].replace('\n', ''))

    # print(correct_output_value)
    
    # 各故障の種類における回路出力と正常な回路出力を比較して、パスフェイル情報を取得
    fault_output_value = [[0 for _ in range(tp_num)] for _ in range(target_fault_num)]  # 故障出力値を格納する二次元リスト fault_output_value[30個ランダムに選んだ故障][テストパターン数]。fault_output_value[2][4]には、ランダムに選んだ故障のうち2番目に選んだ故障が発生した回路にテストパターン4を入力したときの出力値が格納される
    for i in range(tp_num):
        with open(diagnosis_dir + 'fault_output' + str(i), 'r') as f:
            lines = f.readlines()  # fault_output~ファイルから故障出力値の全行を読み込む
            for j in range(target_fault_num):
                fault_output_value[j][i] = lines[j].replace('\n', '')
    
    print(f"fault_output_value{fault_output_value}")
    
    # 故障出力値と正常な回路出力を比較して、パスフェイル情報を取得
    pass_fail = [[0 for _ in range(tp_num)] for _ in range(target_fault_num)]  # パスフェイル情報を格納するリスト。診断を行う際に使用する
    for i in range(target_fault_num):
        with open(diagnosis_dir + 'pass_fail' + str(i), 'w') as f:
            for j in range(tp_num):
                if correct_output_value[j] == fault_output_value[i][j]:
                    pass_fail_value = '0' # pass 正常な回路出力と一致
                    pass_fail[i][j] = 0
                else:
                    pass_fail_value = '1' # fail 故障が発生した回路の出力値と正常な回路出力が一致しない
                    pass_fail[i][j] = 1
                
                f.write(pass_fail_value)

    print(f"pass_fail{pass_fail}")
    
  # 学習済みの機械学習モデルに入力データを与えて、出力を取得する
    # 入力データをファイルから読み込む
    input_data = mk_input_data()

    # 分割モデルの数を取得
    with open(suplit_num_file, 'r') as f:
        num_models = int(f.readline())
    
    # 診断対象の回路における故障信号線の総数を取得
    with open(cir + 'output', 'r') as f:
        line = f.readline().replace(",", "").replace("\n", "")
        global faulet_line_num
        fault_line_num = int(len(line))  # 診断対象の故障信号線の総数
        print("fault_line_num" + str(fault_line_num))

    # 故障の種類ごとに、パスフェイル情報を格納するリストを作成
    model_pass_fail = [[0 for _ in range(fault_line_num)] for _ in range(input_data_num)]  # 故障出力値を格納するリスト fault_output_value[30個ランダムに選んだ故障][テストパターン数]。fault_output_value[2][4]には、ランダムに選んだ故障のうち2番目に選んだ故障が発生した回路にテストパターン4を入力したときの出力値が格納される


    with open(cir + 'pair_list', 'r') as f:
        integration_num = int(f.readline().split()[1])  # 統合数
        # print(integration_num)
        signal_num = int(f.readline().split()[1])  # 信号線の数
        print("signal_num", signal_num)
        lines = [line.split() for line in f.readlines()]  # 故障信号線のペアを取得
        print(lines)
    
    # 信号線ペアのIDをリストに格納
    print("line_id:" + str(integration_num * len(lines)))
    line_id = [0 for _ in range(signal_num - (signal_num%integration_num))]  # 故障信号線のIDを格納するリスト
    for i in range(len(lines) - (signal_num%integration_num)):    # signal_num%integration_numは、信号線が奇数かどうか(integration)を考慮。
        for j in range(integration_num):
            line_id[i*integration_num + j] = int(lines[i][j])  # 故障信号線のIDを取得
    
    # if signal_num % integration_num != 0:  # 信号線の数が奇数の場合、ペアができていない信号線のIDをリストに追加
    #     for i in range(signal_num % integration_num):
    #         line_id[len(lines) - (signal_num % integration_num) + i] = int(lines[len(lines) - (signal_num % integration_num)][i])
    
    print(line_id)
    print(len(line_id))

    # データを何個づつ分割したのかを取得
    with open(suplit_data_num_file, 'r') as f:
        suplit_data_num = int(f.readline().replace("\n", ""))  # データの分割数を取得
        print(suplit_data_num)

    print(f"input_data_num:{input_data_num}")

    a = [0 for _ in range(input_data_num)]  # input_data_num個の要素を持つリストを作成

    for model_id in range(num_models):

        # モデルの読み込み
        with open(model_folder + '/' + cir + 'model_' + str(model_id) + '.tflite', 'rb') as f:  # モデルを読み込むファイルを開く
            tflite_model = f.read()

        # モデルの評価
        interpreter = tf.lite.Interpreter(model_content=tflite_model)  # TFLite形式のモデルを読み込む。保存されたTFLiteモデルをメモリに読み込み、推論を行う準備をするためのインタープリターを作成します。
        interpreter.allocate_tensors()  # #メモリを確保。モデルが使用するテンソル（データ構造）をメモリに割り当てます。TFLiteモデルをインタープリターにロードするだけでは、テンソルのメモリは割り当てられていません。この行を実行することで、モデルが推論に必要なメモリを確保します。

        input_details = interpreter.get_input_details()  # モデルの入力テンソルの詳細を取得。モデルの入力に関する詳細情報をリスト形式で返します。各要素は、入力テンソルの形状、データ型、名前などの情報を含む辞書です。
        output_details = interpreter.get_output_details()  # モデルの出力テンソルの詳細を取得。モデルの出力に関する詳細情報をリスト形式で返します。各要素は、出力テンソルの形状、データ型、名前などの情報を含む辞書です。
        
        n = 0
        for i in range(input_data_num):
            
            input_shape = input_details[0]['shape']  # 入力データの形状を取得.先ほど取得したinput_detailsのリストの中から、形状情報を取得。input_details[0]['shape']は、入力テンソルの形状を取得するためのコードです。['shape']は、辞書内の shape キーを指定しています。

            reshape_input_data = np.reshape(input_data[i], input_shape)  # NumPy配列の形状を指定した形 (input_shape) に変更します。これにより、入力データの形状がモデルの入力テンソルの形状と一致します。


            # モデルの入力データを設定
            interpreter.set_tensor(input_details[0]['index'], np.array(reshape_input_data, dtype=np.float32))   # モデルの入力テンソルにデータを設定します。入力テンソルのインデックス、データを指定します。[0]['index']は、入力テンソルのインデックスを取得するためのコードです。

            # モデルの推論を実行
            interpreter.invoke()

            # モデルの出力データを取得
            output_data = interpreter.get_tensor(output_details[0]['index'])

            correct_file = correct_data_file + str(model_id)  # 分割された正解データファイル名に変更　＝　s344分割正解データ/s344integrated_output + 番号
            mk_output_data(correct_file)  # 関数内でグローバル変数output_node_numを設定

            for j in range(output_node_num):
                if output_data[0][j] < 0.10:
                    output_data[0][j] = 0
                elif output_data[0][j] < 0.5:
                    output_data[0][j] = 0.375
                elif output_data[0][j] < 0.9:
                    output_data[0][j] = 0.625
                else:
                    output_data[0][j] = 1
            
            count = 0  # 統合したものの統合を解くために必要なカウント変数
            if model_id != (num_models - 1):  # 最後のモデル以外の場合
                for j in range(output_node_num):
                    # print(j + count + integration_num*suplit_data_num*model_id)
                    # print(f"line_id: {line_id[j + count + integration_num*suplit_data_num*model_id]}")
                    if output_data[0][j] == 0:
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]= 0
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id + 1]] = 0
                    elif output_data[0][j] == 0.375:
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]= 1
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id + 1]] = 0
                    elif output_data[0][j] == 0.625:
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]= 0
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id + 1]] = 1
                    else:
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]= 1
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id + 1]] = 1
                    # if line_id[j + count + integration_num*suplit_data_num*model_id] == 0:
                    #     a[n] = model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]
                    #     n += 1
                    count += 1
            else:  # 最後のモデルの場合
                for j in range(output_node_num - 1):
                    # print("だｋｆｄｊｊｈだｌｋｈふぃｄぱふぁｄｆだｋｆｄｊｊｈだｌｋｈふぃｄぱふぁｄｆ")
                    # print(j + count + integration_num*suplit_data_num*model_id)
                    # print(f"line_id: {line_id[j + count + integration_num*suplit_data_num*model_id]}")
                    if output_data[0][j] == 0:
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]= 0
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id + 1]] = 0
                    elif output_data[0][j] == 0.375:
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]= 1
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id + 1]] = 0
                    elif output_data[0][j] == 0.625:
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]= 0
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id + 1]] = 1
                    else:
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]= 1
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id + 1]] = 1
                    # if line_id[j + count + integration_num*suplit_data_num*model_id] == 183:
                    #     a[n] = model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]
                    #     n += 1
                    count += 1


        
            # print(output_data)                # output_dataには、テストパターンnのときの、故障があるかどうかの情報が格納されている

    # print(model_pass_fail[0])
    # print(model_pass_fail[1])
    # print(model_pass_fail[2])
    # print(model_pass_fail[3])
    # print(a)


  # モデル出力を故障の種類（fault_type_sum個）に分ける
    # model_pass_failは、行はモデルの入力データ数、列は故障信号線の数を表す二次元リストであるため、パスフェイル情報（pass_failnファイル）と比較するために、行と列を入れ替える
    model_pass_fail = [[row[i] for row in model_pass_fail] for i in range(len(model_pass_fail[0]))]  #内包表記を使って行と列を入れ替える

    compare_model_pass_fail = [[0 for _ in range(tp_num)] for _ in range(signal_num*fault_type_sum)] # 各行が各故障が起きた場合の各テストパターンのパスフェイル情報を格納する二次元リスト
    
    for i in range(signal_num):
        for j in range(fault_type_sum):
            for k in range(tp_num):
                compare_model_pass_fail[i*fault_type_sum + j][k] = model_pass_fail[i][k*fault_type_sum + j] 

    print("compare_model_pass_fail", compare_model_pass_fail)

    
    st_compare_model_pass_fail = [[0 for _ in range(tp_num)] for _ in range(signal_num*2)] # 縮退故障用のパスフェイルリスト。各行が各故障が起きた場合の各テストパターンのパスフェイル情報を格納する二次元リスト
    br_compare_model_pass_fail = [[0 for _ in range(tp_num)] for _ in range(signal_num*(fault_type_sum - 2))] # ブリッジ故障用のパスフェイルリスト。各行が各故障が起きた場合の各テストパターンのパスフェイル情報を格納する二次元リスト

    st_idx = 0
    br_idx = 0
    for i in range(signal_num*fault_type_sum):
        if (i%fault_type_sum == 0) or (i%fault_type_sum == 1): # 縮退故障の場合
            for j in range(tp_num):
                st_compare_model_pass_fail[st_idx][j] = compare_model_pass_fail[i][j]
            st_idx += 1
        else: # ブリッジ故障の場合
            for j in range(tp_num):
                br_compare_model_pass_fail[br_idx][j] = compare_model_pass_fail[i][j]
            br_idx += 1
    
    print("st_compare_model_pass_fail", st_compare_model_pass_fail)
    print("br_compare_model_pass_fail", br_compare_model_pass_fail)
    
# ランダムに選んだ故障のパスフェイル情報（pass_failリスト）とcompare_model_pass_failを比較して、故障候補を取得する
    # 縮退故障のパスフェイル情報を比較
    st_fault_candidate = [[] for _ in range(target_fault_num)] # 故障候補を格納するリスト
    st_fault_type = [[] for _ in range(target_fault_num)] # 故障の種類を格納するリスト 0=0縮退故障、1=1縮退故障、2=ブリッジ故障➀、3=ブリッジ故障➁、4=ブリッジ故障➂、5=ブリッジ故障➃、6=ブリッジ故障➄、7=ブリッジ故障➅、8=ブリッジ故障➆、9=ブリッジ故障➇、10=ブリッジ故障➈、11=ブリッジ故障➉
    print(len(st_fault_all_line))
    for i in range(target_fault_num):
        line_count = 0
        count = 0
        for j in range(signal_num*2):
            if st_compare_model_pass_fail[j] == pass_fail[i]:  # パスフェイル情報を比較
                # print(line_count, len(st_fault_candidate[i]), i, j, count)
                st_fault_candidate[i].append(st_fault_all_line[line_count])  # 故障候補を格納するリストに追加
                st_fault_type[i].append(count%2)  # 故障の種類を格納するリストに追加
            
            count += 1
            if count == 2:  # st_compare_model_pass_failは、2行ごとが0または1縮退故障の信号線に対応しているため、2行ごとにカウントをリセットする
                line_count += 1
                count = 0


    # 故障候補の中に、実際に縮退故障として選んだ信号線と故障の種類が一致するものがあるかどうかを確認
    find_count = 0  # 見つけられた故障の数
    not_find_count = 0  # 見つけられなかった故障の数
    with open(diagnosis_dir + 'correct_fault_candidate', 'w') as f:
        with open(diagnosis_dir + 'incorrect_fault_candidate', 'w') as g:
            for i in range(target_fault_num):
                st_fault_candidate[i] = sorted(st_fault_candidate[i]) # 小さい番号順に並び替える
                for j in range(len(st_fault_candidate[i])):
                    if st_fault_candidate[i][j] == st_fault_target_line[i] and st_fault_type[i][j] == st_fault_target_type[i]:
                        count += 1
                        print(f"故障候補の中に実際に選んだ故障 {st_fault_candidate[i][j]} の {st_fault_type[i][j]} 縮退故障が含まれていました")
                        f.write("故障候補数；" + str(len(st_fault_candidate[i])) + '\n')
                        f.write(str(st_fault_candidate[i][j]) + ' sa ' + str(st_fault_type[i][j]) + '\n')
                        break
                    if j == len(st_fault_candidate[i]) - 1:
                        print(f"故障候補の中に実際に選んだ故障 {st_fault_candidate[i][j]} の {st_fault_type[i][j]} 縮退故障は含まれていませんでした")
                        g.write("見つけられなかった故障：" + str(st_fault_candidate[i][j]) + ' sa ' + str(st_fault_type[i][j]) + '\n')
                        not_find_count += 1


    print("見つけられた故障の数：" + str(count))
    print("見つけられなかった故障の数：" + str(not_find_count))
    print("縮退故障の診断率：" + str((count/target_fault_num)*100) + "%")





22
670
['id 0 Fault 1 sa 0\n', '11111111100111000010001111\n', 'id 1 Fault 1 sa 1\n', '11111111100000111110001111\n', 'id 2 Fault 2 sa 0\n', '11111111100000111110001111\n', 'id 3 Fault 2 sa 1\n', '11111111100000111110001111\n', 'id 4 Fault 3 sa 0\n', '11111111100000111110001111\n', 'id 5 Fault 3 sa 1\n', '11111111100000111110001111\n', 'id 6 Fault 4 sa 0\n', '11111111100000111110001111\n', 'id 7 Fault 4 sa 1\n', '11111111100000111110001111\n', 'id 8 Fault 5 sa 0\n', '11111111100000111110001111\n', 'id 9 Fault 5 sa 1\n', '11111111100000111110001111\n', 'id 10 Fault 6 sa 0\n', '11111111100000111110001111\n', 'id 11 Fault 6 sa 1\n', '11111111100000111110001111\n', 'id 12 Fault 7 sa 0\n', '11111111100000111110001111\n', 'id 13 Fault 7 sa 1\n', '11111111100000111110001111\n', 'id 14 Fault 8 sa 0\n', '11111111100000111110001111\n', 'id 15 Fault 8 sa 1\n', '11111111100000111110001111\n', 'id 16 Fault 9 sa 0\n', '11111111100000111110001111\n', 'id 17 Fault 9 sa 1\n', '1111111110000011111000111

In [ ]:
2025/05/30

# 学習済みのANNを用いて学習を行うプログラム
# cd workspace/research2/experiment
#　実行コマンド：　python3 diagnosis.py

import numpy as np
import logging
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential, model_from_json
from keras.layers import Dense, Dropout, BatchNormalization, Activation
from keras.layers import LeakyReLU
from tensorflow.keras.utils import get_custom_objects
from keras import optimizers
from keras import backend as K
from tensorflow.keras.optimizers import Adam

import matplotlib
import matplotlib.pyplot as plt
from memory_profiler import memory_usage
import time
import pathlib
import sklearn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix

import sys
import argparse
import os
import multiprocessing
from multiprocessing import Process

import random

#グローバル変数
cir = 's344'  # 対象回路
tp_file = cir + '.vec'  # テストパターンファイル名
part_stdic_file = cir + "stdic_bi/aout_" #縮退故障辞書ファイル名の一部
part_brdic_file = cir + "brdic_bi/aout_" #ブリッジ故障辞書ファイルの一部
st_diagnosis_dir = cir + "diagnosis_st_data/" #縮退故障診断を行うためのデータを保存するフォルダ
br_diagnosis_dir = cir + "diagnosis_br_data/" #ブリッジ故障診断を行うためのデータを保存するフォルダ
target_fault_num = 30  # 故障診断対象の故障の数⇒全ての故障の中からtarget_fault_num個をランダムに選択する
correct_output_file = 'correct_output/' + cir + '_correct_output'  # 正常な回路の出力ファイル
input_data_file = cir + 'input'  # 入力データファイル
correct_data_file = cir + '分割正解データ' + '/' + cir + 'integrated_output'  # 統合後の正解データファイル
suplit_num_file = cir + '分割正解データ' + '/' + cir + 'suplit_num'  # モデルの分割数が保存されたファイル
suplit_data_num_file = cir + '分割正解データ' + '/' + cir + 'suplit_data_num'  # データの分割数が保存されたファイル
input_data_num = None  # 1個のモデルにおける学習データ数
input_node_num = None  #入力層におけるノード数 初期値は8　学習結果によって変更
output_node_num = None #　出力層のノード数＝分割数による
num_models = None  # 学習させるモデルの数
model_folder = cir + 'sepmodel'  # 学習済みモデルを保存するフォルダ
fault_line_num = None  # 故障診断対象の回路における故障信号線の総数
fault_type_sum = 12  # 故障の種類の総数

processes = 8  # 並列処理のプロセス数

def mk_input_data():
    # 入力データファイルを開いてデータを読み込む
    with open(input_data_file) as f:
        lines = [_.replace(",", "").replace("\n", "") for _ in f.readlines()]

    # print(lines)
    
    global input_data_num
    input_data_num = int(len(lines))      #学習データ数を設定。学習データ数は入力データの行数
    global input_node_num               #グローバル変数を書き換え
    input_node_num = int(len(lines[0]))      #入力ノード数を設定。入力ノード数は入力データの各行の要素数

    int_lines = [list(map(int, _)) for _ in lines]  #list要素の型をint型に変換

    return np.array(int_lines)


def mk_output_data(fname):
    # 正解データファイルを開いてデータを読み込む
    with open(fname) as f:
        lines = [_.replace("\n", "") for _ in f.readlines()]
    
    lines = [value.split(",") for value in lines] # カンマ区切りで各信号線をリストに格納
    
    # print("dsadsa")
    # print(lines[0])

    for i in range(len(lines)):
        for j in range(len(lines[i])):
            lines[i][j] = float(lines[i][j])
    
    # print("gaga")
    # print(lines[0])
    
    global output_node_num               #グローバル変数を書き換え
    output_node_num = int(len(lines[0]))      #出力ノード数を設定。出力ノード数は正解データの各行の要素数


# 故障が発生する信号線をtarget_fault_num個ランダムに選択
def get_fault_target(fault_all_line, fault_flag):  # fault_flagは0なら縮退故障、1ならブリッジ故障を表す
    fault_target_line = random.sample(fault_all_line, target_fault_num) # 想定する故障信号線をランダムに選ぶ
    fault_target_line = sorted(fault_target_line) # 小さい番号順に並び替える

    if fault_flag == 0:  # 縮退故障の診断を行う場合
        fault_target_type = random.choices([0, 1], k=target_fault_num)  # 0縮退故障か1縮退故障かをランダムに決定する
        with open(st_diagnosis_dir + 'st_fault_line', 'w') as f: # 診断対象故障信号線番号をファイルに保存
            f.write('診断対象縮退故障数' + str(target_fault_num) + '\n')  # 故障の数をファイルに保存
            for i in range(target_fault_num):
                f.write(str(fault_target_line[i]) + ' ' + "sa " + str(fault_target_type[i]) + "\n")
    else:  # ブリッジ故障の診断を行う場合
        fault_target_type = random.choices(list(range(10)), k=target_fault_num)  # ブリッジ故障の種類(10個の中から)をランダムに決定する。0~9の整数値をランダムに選ぶ
        with open(br_diagnosis_dir + 'br_fault_line', 'w') as f:  # 診断対象故障信号線番号をファイルに保存
            f.write('診断対象ブリッジ故障数' + str(target_fault_num) + '\n')  # 故障の数をファイルに保存
            for i in range(target_fault_num):
                f.write(str(fault_target_line[i]) + ' ' + "br_type " + str(fault_target_type[i]) + "\n")

    print(fault_target_line)
    print(fault_target_type)

    return fault_target_line, fault_target_type


# 故障出力値を取得してファイルに保存する関数
def get_fault_output(tp_num, fault_target_line, fault_target_type, fault_flag): # fault_flagは0なら縮退故障、1ならブリッジ故障を表す
    # 縮退故障、ブリッジ故障の辞書ファイルのパスを指定
    if fault_flag == 0:  # 縮退故障の診断を行う場合
        part_dic_file = part_stdic_file  # 縮退故障辞書ファイルのパスを指定
        diagnosis_file = st_diagnosis_dir + 'fault_output'  # 縮退故障診断用の故障出力保存ファイル名
    else:  # ブリッジ故障の診断を行う場合
        part_dic_file = part_brdic_file  # ブリッジ故障辞書ファイルのパスを指定
        diagnosis_file = br_diagnosis_dir + 'fault_output'  # ブリッジ故障診断用の故障出力保存ファイル名

    # 診断対象故障の出力値をファイルに保存
    for i in range(tp_num):
        with open(part_dic_file + str(i), 'r') as f:    # 故障辞書ファイルを指定
            fault_inf = f.readline()  # 故障情報
            lines = f.readlines()  # 故障辞書ファイルの全行を読み込む

        # 各辞書におけるIDと出力値を取得
        fault_output = [0 for _ in range(target_fault_num)]  # 故障出力値を格納するリスト
        idx = 0
        br_type_count = 0  # ブリッジ故障の種類をカウントする変数
        for j in range(0, len(lines), 2):
            if fault_flag == 0:  # 縮退故障の診断を行う場合
                if fault_target_line[idx] == int(lines[j].split()[3]) and fault_target_type[idx] == int(lines[j].split()[5]): # ファイルの「id 0 Fault 1 sa 0」の行から、信号線番号と0、1縮退故障情報を取得して、比較
                    fault_output[idx] = lines[j+1].replace('\n', '')  # 故障出力値を取得
                    idx += 1
            else:  # ブリッジ故障の診断を行う場合
                if fault_target_line[idx] == int(lines[j].split()[3]) and fault_target_type[idx] == (br_type_count%10):  # ファイルの「id 0 Br_flt 1 1 2」の行から、信号線番号を取得して、br_type_countの値から、10種類あるブリッジ故障の種類を計算して、比較
                    fault_output[idx] = lines[j+1].replace('\n', '')  # 故障出力値を取得
                    idx += 1
                br_type_count += 1  # ブリッジ故障の種類をカウントする
                if br_type_count == 10:
                    br_type_count = 0

            if idx == target_fault_num:  # 診断対象の故障数に達したら、ループを抜ける。これは、診断対象の故障信号線が小さい順に並び替えられているため、target_fault_num個の故障を取得したら、ループを抜ける
                break

        with open(diagnosis_file + str(i), 'w') as f:
            for j in range(len(fault_output)):
                f.write(fault_output[j] + '\n')
    
    print(f"fault_output：{fault_output}")


# パスフェイル情報を取得してファイルに保存する関数
def get_pass_fail(tp_num, fault_flag):  # fault_flagは0なら縮退故障、1ならブリッジ故障を表す
    if fault_flag == 0:  # 縮退故障の診断を行う場合
        fault_output_file = st_diagnosis_dir + 'fault_output'  # 縮退故障の故障出力を保存しているファイルのパスを指定
        pass_fail_file = st_diagnosis_dir + 'fault_output'  # 縮退故障のパスフェイル保存ファイル名
    else:  # ブリッジ故障の診断を行う場合
        fault_output_file = br_diagnosis_dir + 'fault_output'  # ブリッジ故障の故障出力を保存しているファイルのパスを指定
        pass_fail_file = br_diagnosis_dir + 'pass_fail'  # ブリッジ故障のパスフェイル保存ファイル名


    #正常な回路出力を取得する
    correct_output_value = []  # 正常な回路出力を格納するリスト
    with open(correct_output_file, 'r') as f:
        lines = f.readlines()  # 正常な回路出力の全行を読み込む
        for i in range(1, len(lines), 2):
            correct_output_value.append(lines[i].replace('\n', ''))

    # print(correct_output_value)
    
    # 各故障の種類における回路出力と正常な回路出力を比較して、パスフェイル情報を取得
    fault_output_value = [[0 for _ in range(tp_num)] for _ in range(target_fault_num)]  # 故障出力値を格納する二次元リスト fault_output_value[30個ランダムに選んだ故障][テストパターン数]。fault_output_value[2][4]には、ランダムに選んだ故障のうち2番目に選んだ故障が発生した回路にテストパターン4を入力したときの出力値が格納される
    for i in range(tp_num):
        with open(fault_output_file + str(i), 'r') as f:
            lines = f.readlines()  # fault_output~ファイルから故障出力値の全行を読み込む
            for j in range(target_fault_num):
                fault_output_value[j][i] = lines[j].replace('\n', '')
    
    print(f"fault_output_value{fault_output_value}")
    
    # 故障出力値と正常な回路出力を比較して、パスフェイル情報を取得
    pass_fail = [[0 for _ in range(tp_num)] for _ in range(target_fault_num)]  # パスフェイル情報を格納するリスト。診断を行う際に使用する
    for i in range(target_fault_num):
        with open(pass_fail_file + str(i), 'w') as f:
            for j in range(tp_num):
                if correct_output_value[j] == fault_output_value[i][j]:
                    pass_fail_value = '0' # pass 正常な回路出力と一致
                    pass_fail[i][j] = 0
                else:
                    pass_fail_value = '1' # fail 故障が発生した回路の出力値と正常な回路出力が一致しない
                    pass_fail[i][j] = 1
                
                f.write(pass_fail_value)

    print(f"pass_fail{pass_fail}")

    return pass_fail  # パスフェイル情報を返す


#  故障候補を取得する関数
def get_fault_candidate(pass_fail, compare_model_pass_fail, fault_all_line, signal_num, br_missing_line, fault_flag): # fault_flagは0なら縮退故障、1ならブリッジ故障を表す
    if fault_flag == 0:  # 縮退故障の診断を行う場合
        fault_type_num = 2  # 縮退故障の種類は0縮退故障と1縮退故障の2種類
    else:  # ブリッジ故障の診断を行う場合
        fault_type_num = 10  # ブリッジ故障の種類は0~9の10種類
        signal_num = signal_num - len(br_missing_line)  # ブリッジ故障の診断を行う場合、ブリッジ故障が発生しない信号線を除外するため、signal_numを更新する


    # 縮退故障のパスフェイル情報を比較
    fault_candidate = [[] for _ in range(target_fault_num)] # 故障候補を格納するリスト
    fault_type = [[] for _ in range(target_fault_num)] # 縮退故障の種類を格納するリスト 縮退故障の場合（0=0縮退故障、1=1縮退故障）。ブリッジ故障の場合（0~9=ブリッジ故障の種類）
    print(f"fault_all_line{len(fault_all_line)}")
    for i in range(target_fault_num):
        line_count = 0
        count = 0
        for j in range(signal_num*fault_type_num):  # signal_numは、診断対象の回路における信号線の総数。fault_type_numは、縮退故障の場合は2、ブリッジ故障の場合は10
            if compare_model_pass_fail[j] == pass_fail[i]:  # パスフェイル情報を比較
                # print(line_count, len(st_fault_candidate[i]), i, j, count)
                fault_candidate[i].append(fault_all_line[line_count])  # 故障候補を格納するリストに追加
                fault_type[i].append(count%fault_type_num)  # 故障の種類を格納するリストに追加
            
            count += 1
            if count == fault_type_num:  # st_compare_model_pass_failは、2行ごとが0または1縮退故障の信号線に対応しているため、2行ごとにカウントをリセットする
                line_count += 1
                count = 0
    
    return fault_candidate, fault_type  # 故障候補と故障の種類を返す


# 故障候補の中に、診断対象の故障信号線が含まれているかを確認する関数
def check_fault_candidate(fault_candidate, fault_type, fault_target_line, fault_target_type, fault_flag): # fault_flagは0なら縮退故障、1ならブリッジ故障を表す
    if fault_flag == 0:  # 縮退故障の診断を行う場合
        correct_fault_candidate_file = st_diagnosis_dir + 'correct_fault_candidate'  # 正しい故障候補を保存するファイル名
        incorrect_fault_candidate_file = st_diagnosis_dir + 'incorrect_fault_candidate'  # 間違った故障候補を保存するファイル名
        diagnosis_rate_file = st_diagnosis_dir + 'diagnosis_rate'  # 診断率を保存するファイル名
        fault_name = '縮退故障'  # 故障の名前
        fault_symbol = ' sa '  # 縮退故障の記号
    else:  # ブリッジ故障の診断を行う場合
        correct_fault_candidate_file = br_diagnosis_dir + 'correct_fault_candidate'  # 正しい故障候補を保存するファイル名
        incorrect_fault_candidate_file = br_diagnosis_dir + 'incorrect_fault_candidate'  # 間違った故障候補を保存するファイル名
        diagnosis_rate_file = br_diagnosis_dir + 'diagnosis_rate'  # 診断率を保存するファイル名
        fault_name = 'ブリッジ故障'  # 故障の名前
        fault_symbol = ' br_type '  # ブリッジ故障の記号

    find_count = 0  # 見つけられた故障の数
    not_find_count = 0  # 見つけられなかった故障の数
    with open(correct_fault_candidate_file, 'w') as f:
        with open(incorrect_fault_candidate_file, 'w') as g:
            for i in range(target_fault_num):
                fault_candidate[i] = sorted(fault_candidate[i]) # 小さい番号順に並び替える
                for j in range(len(fault_candidate[i])):
                    if fault_candidate[i][j] == fault_target_line[i] and fault_type[i][j] == fault_target_type[i]:
                        find_count += 1
                        print(f"{fault_name}候補の中に実際に選んだ{fault_name}番号 {fault_candidate[i][j]} の {fault_type[i][j]} {fault_name}が含まれていました")
                        f.write("故障候補数；" + str(len(fault_candidate[i])) + '\n')
                        f.write(str(fault_candidate[i][j]) + fault_symbol + str(fault_type[i][j]) + '\n')
                        break
                    if j == len(fault_candidate[i]) - 1:
                        print(f"故障候補の中に実際に選んだ故障 {fault_candidate[i][j]} の {fault_type[i][j]} 縮退故障は含まれていませんでした")
                        g.write("見つけられなかった故障：" + str(fault_candidate[i][j]) + fault_symbol + str(fault_type[i][j]) + '\n')
                        not_find_count += 1

    print("見つけられた故障の数：" + str(find_count))
    print("見つけられなかった故障の数：" + str(not_find_count))
    print(fault_name + "の診断率：" + str((find_count/target_fault_num)*100) + "%")

    import datetime as dt
    datetime = dt.datetime.now()
    with open(diagnosis_rate_file, 'a') as f:
        f.write("\n")
        f.write(str(datetime) + '\n')
        f.write("見つけられた故障の数：" + str(find_count) + '\n')
        f.write("見つけられなかった故障の数：" + str(not_find_count) + '\n')
        f.write(fault_name + "の診断率：" + str((find_count/target_fault_num)*100) + "%\n")



if __name__ == '__main__':
    
    # テストパターン数を取得
    with open(tp_file, 'r') as f:
        line = f.readline()
        tp_num = int(line.split()[0])  # テストパターン数
        print(tp_num)

    # 故障診断対象の回路における「縮退故障」の総数を取得
    with open(part_stdic_file + str(0), 'r') as f:  # 縮退故障辞書ファイルの1つを開く
        fault_inf = f.readline()  # 故障情報
        st_fault_num = int(fault_inf.split()[2])  # 対象回路で起こりうる故障数
        lines = f.readlines()[::2]  # 故障辞書ファイルのid情報を読み込む
        st_fault_all_line = [int(line.split()[3]) for line in lines[::2]]  # 縮退故障信号線番号を取得
        print(st_fault_num)
        print("st_fault_all_line：", st_fault_all_line)
        # print(suplit_num)

    # 故障診断対象の回路における「ブリッジ故障」の総数を取得
    with open(part_brdic_file + str(0), 'r') as f:  # ブリッジ故障辞書ファイルの1つを開く
        fault_inf = f.readline()  # 故障情報
        br_fault_num = int(fault_inf.split()[2]) # 対象回路で起こりうる故障数
        lines = f.readlines()[::2]  # 故障辞書ファイルのid情報を読み込む
        br_fault_all_line = [int(line.split()[3]) for line in lines[::10]]  # ブリッジ故障が発生する可能性がある信号線番号（支配される信号線の番号）をリストに格納　※縮退故障と異なり、出力信号線以外にもブリッジ故障が発生しない信号線があるため、故障辞書から読み込まないといけない
        # br_fault_type = [int(line.split()[4]) for line in lines[::2]] # ブリッジ故障における故障の種類（0、1どちらに支配されるのか）を取得
        # br_dominate_line = [int(line.split()[5]) for line in lines[::2]]  # 支配する信号線を格納
        print(br_fault_num)
        print("br_fault_all_line：", br_fault_all_line)
    

    # 診断対象となる縮退故障、ブリッジ故障それぞれに対する対象信号線番号を30個ランダムにそれぞれ取得
    st_fault_target_line, st_fault_target_type = get_fault_target(st_fault_all_line, 0) # 縮退故障が発生する信号線をtarget_fault_num個ランダムに選択。さらに、その信号線で発生するのは0縮退故障か1縮退故障かをランダムに決定する
    br_fault_target_line, br_fault_target_type = get_fault_target(br_fault_all_line, 1) # ブリッジ故障が発生する信号線をtarget_fault_num個ランダムに選択。さらに、その信号線で発生するのは0縮退故障か1縮退故障かをランダムに決定する



    # 診断対象故障の出力値をファイルに保存
    get_fault_output(tp_num, st_fault_target_line, st_fault_target_type, 0) # 縮退故障時の故障出力値をファイルに保存
    get_fault_output(tp_num, br_fault_target_line, br_fault_target_type, 1) # ブリッジ故障時の故障出力値をファイルに保存


    # パスフェイル情報を取得してファイルに保存
    st_pass_fail = get_pass_fail(tp_num, 0)  # 縮退故障のパスフェイル情報を取得してファイルに保存
    br_pass_fail = get_pass_fail(tp_num, 1)  # ブリッジ故障のパスフェイル情報を取得してファイルに保存
    
    
  # 学習済みの機械学習モデルに入力データを与えて、出力を取得する
    # 入力データをファイルから読み込む
    input_data = mk_input_data()

    # 分割モデルの数を取得
    with open(suplit_num_file, 'r') as f:
        num_models = int(f.readline())
    
    # 診断対象の回路における故障信号線の総数を取得
    with open(cir + 'output', 'r') as f:
        line = f.readline().replace(",", "").replace("\n", "")
        global faulet_line_num
        fault_line_num = int(len(line))  # 診断対象の故障信号線の総数
        print("fault_line_num" + str(fault_line_num))

    # 故障の種類ごとに、パスフェイル情報を格納するリストを作成
    model_pass_fail = [[0 for _ in range(fault_line_num)] for _ in range(input_data_num)]  # 故障出力値を格納するリスト fault_output_value[30個ランダムに選んだ故障][テストパターン数]。fault_output_value[2][4]には、ランダムに選んだ故障のうち2番目に選んだ故障が発生した回路にテストパターン4を入力したときの出力値が格納される


    with open(cir + 'pair_list', 'r') as f:
        integration_num = int(f.readline().split()[1])  # 統合数
        # print(integration_num)
        signal_num = int(f.readline().split()[1])  # 信号線の数
        print("signal_num", signal_num)
        lines = [line.split() for line in f.readlines()]  # 故障信号線のペアを取得
        print(lines)
    
    # 信号線ペアのIDをリストに格納
    print("line_id:" + str(integration_num * len(lines)))
    line_id = [0 for _ in range(signal_num - (signal_num%integration_num))]  # 故障信号線のIDを格納するリスト
    for i in range(len(lines) - (signal_num%integration_num)):    # signal_num%integration_numは、信号線が奇数かどうか(integration)を考慮。
        for j in range(integration_num):
            line_id[i*integration_num + j] = int(lines[i][j])  # 故障信号線のIDを取得
    
    # if signal_num % integration_num != 0:  # 信号線の数が奇数の場合、ペアができていない信号線のIDをリストに追加
    #     for i in range(signal_num % integration_num):
    #         line_id[len(lines) - (signal_num % integration_num) + i] = int(lines[len(lines) - (signal_num % integration_num)][i])
    
    print(line_id)
    print(len(line_id))

    # データを何個づつ分割したのかを取得
    with open(suplit_data_num_file, 'r') as f:
        suplit_data_num = int(f.readline().replace("\n", ""))  # データの分割数を取得
        print(suplit_data_num)

    print(f"input_data_num:{input_data_num}")

    a = [0 for _ in range(input_data_num)]  # input_data_num個の要素を持つリストを作成

    for model_id in range(num_models):

        # モデルの読み込み
        with open(model_folder + '/' + cir + 'model_' + str(model_id) + '.tflite', 'rb') as f:  # モデルを読み込むファイルを開く
            tflite_model = f.read()

        # モデルの評価
        interpreter = tf.lite.Interpreter(model_content=tflite_model)  # TFLite形式のモデルを読み込む。保存されたTFLiteモデルをメモリに読み込み、推論を行う準備をするためのインタープリターを作成します。
        interpreter.allocate_tensors()  # #メモリを確保。モデルが使用するテンソル（データ構造）をメモリに割り当てます。TFLiteモデルをインタープリターにロードするだけでは、テンソルのメモリは割り当てられていません。この行を実行することで、モデルが推論に必要なメモリを確保します。

        input_details = interpreter.get_input_details()  # モデルの入力テンソルの詳細を取得。モデルの入力に関する詳細情報をリスト形式で返します。各要素は、入力テンソルの形状、データ型、名前などの情報を含む辞書です。
        output_details = interpreter.get_output_details()  # モデルの出力テンソルの詳細を取得。モデルの出力に関する詳細情報をリスト形式で返します。各要素は、出力テンソルの形状、データ型、名前などの情報を含む辞書です。
        
        n = 0
        for i in range(input_data_num):
            
            input_shape = input_details[0]['shape']  # 入力データの形状を取得.先ほど取得したinput_detailsのリストの中から、形状情報を取得。input_details[0]['shape']は、入力テンソルの形状を取得するためのコードです。['shape']は、辞書内の shape キーを指定しています。

            reshape_input_data = np.reshape(input_data[i], input_shape)  # NumPy配列の形状を指定した形 (input_shape) に変更します。これにより、入力データの形状がモデルの入力テンソルの形状と一致します。


            # モデルの入力データを設定
            interpreter.set_tensor(input_details[0]['index'], np.array(reshape_input_data, dtype=np.float32))   # モデルの入力テンソルにデータを設定します。入力テンソルのインデックス、データを指定します。[0]['index']は、入力テンソルのインデックスを取得するためのコードです。

            # モデルの推論を実行
            interpreter.invoke()

            # モデルの出力データを取得
            output_data = interpreter.get_tensor(output_details[0]['index'])

            correct_file = correct_data_file + str(model_id)  # 分割された正解データファイル名に変更　＝　s344分割正解データ/s344integrated_output + 番号
            mk_output_data(correct_file)  # 関数内でグローバル変数output_node_numを設定

            for j in range(output_node_num):
                if output_data[0][j] < 0.25:
                    output_data[0][j] = 0
                elif output_data[0][j] < 0.5:
                    output_data[0][j] = 0.375
                elif output_data[0][j] < 0.75:
                    output_data[0][j] = 0.625
                else:
                    output_data[0][j] = 1
            
            count = 0  # 統合したものの統合を解くために必要なカウント変数
            if model_id != (num_models - 1):  # 最後のモデル以外の場合
                for j in range(output_node_num):
                    # print(j + count + integration_num*suplit_data_num*model_id)
                    # print(f"line_id: {line_id[j + count + integration_num*suplit_data_num*model_id]}")
                    if output_data[0][j] == 0:
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]= 0
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id + 1]] = 0
                    elif output_data[0][j] == 0.375:
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]= 1
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id + 1]] = 0
                    elif output_data[0][j] == 0.625:
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]= 0
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id + 1]] = 1
                    else:
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]= 1
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id + 1]] = 1
                    # if line_id[j + count + integration_num*suplit_data_num*model_id] == 0:
                    #     a[n] = model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]
                    #     n += 1
                    count += 1
            else:  # 最後のモデルの場合
                for j in range(output_node_num - 1):
                    # print("だｋｆｄｊｊｈだｌｋｈふぃｄぱふぁｄｆだｋｆｄｊｊｈだｌｋｈふぃｄぱふぁｄｆ")
                    # print(j + count + integration_num*suplit_data_num*model_id)
                    # print(f"line_id: {line_id[j + count + integration_num*suplit_data_num*model_id]}")
                    if output_data[0][j] == 0:
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]= 0
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id + 1]] = 0
                    elif output_data[0][j] == 0.375:
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]= 1
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id + 1]] = 0
                    elif output_data[0][j] == 0.625:
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]= 0
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id + 1]] = 1
                    else:
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]= 1
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id + 1]] = 1
                    # if line_id[j + count + integration_num*suplit_data_num*model_id] == 183:
                    #     a[n] = model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]
                    #     n += 1
                    count += 1


        
            # print(output_data)                # output_dataには、テストパターンnのときの、故障があるかどうかの情報が格納されている

    # print(model_pass_fail[0])
    # print(model_pass_fail[1])
    # print(model_pass_fail[2])
    # print(model_pass_fail[3])
    # print(a)


  # モデル出力を故障の種類（fault_type_sum個）に分ける
    # model_pass_failは、行はモデルの入力データ数、列は故障信号線の数を表す二次元リストであるため、パスフェイル情報（pass_failnファイル）と比較するために、行と列を入れ替える
    model_pass_fail = [[row[i] for row in model_pass_fail] for i in range(len(model_pass_fail[0]))]  #内包表記を使って行と列を入れ替える

    compare_model_pass_fail = [[0 for _ in range(tp_num)] for _ in range(signal_num*fault_type_sum)] # 各行が各故障が起きた場合の各テストパターンのパスフェイル情報を格納する二次元リスト
    
    for i in range(signal_num):
        for j in range(fault_type_sum):
            for k in range(tp_num):
                compare_model_pass_fail[i*fault_type_sum + j][k] = model_pass_fail[i][k*fault_type_sum + j] 

    print("compare_model_pass_fail", compare_model_pass_fail)

    
    st_compare_model_pass_fail = [[0 for _ in range(tp_num)] for _ in range(signal_num*2)] # 縮退故障用のパスフェイルリスト。各行が各故障が起きた場合の各テストパターンのパスフェイル情報を格納する二次元リスト
    br_compare_model_pass_fail = [[0 for _ in range(tp_num)] for _ in range(signal_num*(fault_type_sum - 2))] # ブリッジ故障用のパスフェイルリスト。各行が各故障が起きた場合の各テストパターンのパスフェイル情報を格納する二次元リスト

    st_idx = 0
    br_idx = 0
    for i in range(signal_num*fault_type_sum):
        if (i%fault_type_sum == 0) or (i%fault_type_sum == 1): # 縮退故障の場合
            for j in range(tp_num):
                st_compare_model_pass_fail[st_idx][j] = compare_model_pass_fail[i][j]
            st_idx += 1
        else: # ブリッジ故障の場合
            for j in range(tp_num):
                br_compare_model_pass_fail[br_idx][j] = compare_model_pass_fail[i][j]
            br_idx += 1
    
    # print("st_compare_model_pass_fail", st_compare_model_pass_fail)
    # print("br_compare_model_pass_fail", br_compare_model_pass_fail)

    with open("a", 'w') as f:
        for i in range(len(br_compare_model_pass_fail)):
            f.write(str(br_compare_model_pass_fail[i]) + '\n')


    # ブリッジ故障においては、縮退故障と異なり、出力信号線以外にもブリッジ故障が生じない信号線があり、ファイルから読み取るしかない。
    # br_fault_all_lineから出力信号線以外で故障が発生しない信号線を特定する
    br_fault_all_line = sorted(br_fault_all_line)  # 縮退故障信号線番号を小さい順に並び替える

    with open("c" + cir, 'r') as f:
        line = f.readline()
        all_signal_num  = int(line.split()[0]) # 出力信号線も含めた全信号線の総数を取得
        cir_output_line_num = int(line.split()[1])  # 出力信号線の総数を取得
        cir_input_line_num = int(line.split()[2])  # 入力信号線の総数を取得
    
    # br_fault_all_lineの中に含まれていない信号線番号を取得
    br_missing_line = [line for line in range(1, (all_signal_num + 1)) if line not in br_fault_all_line]
    # print("br_missing_line：", br_missing_line)  # ブリッジ故障が発生しない信号線番号を表示

    # br_missing_lineには、出力信号線の番号も含まれているため、削除する
    for i in range(cir_output_line_num):
        br_missing_line.remove(i + cir_input_line_num + 1)  # 出力信号線の番号を削除
    
    # print("br_compare_model_pass_failの長さ：", len(br_compare_model_pass_fail))  # ブリッジ故障のパスフェイル情報の長さを表示
    # print("br_fault_all_line：", br_fault_all_line)  # ブリッジ故障が発生する信号線番号を表示
    print("br_missing_line：", br_missing_line)  # ブリッジ故障が発生しない信号線番号を表示
    print("br_missing_lineの長さ：", len(br_missing_line))  # ブリッジ故障が発生しない信号線番号の長さを表示

    # br_compare_model_pass_failから、ブリッジ故障が発生しない信号線に対応しているパスフェイル情報を削除する
    # （注意）br_compare_model_pass_failには、出力信号線は含まれていない。インデックスが小さいものから、入力信号線、入力信号線と出力信号線以外の信号線の順番になっている。
    delite_count = 0  # 削除した数をカウントする変数。リストの行を削除すると、その分繰り下がって行がずれるため、削除した数をカウントしておく必要がある
    for i in range(len(br_missing_line)):
        if br_missing_line[i] <= cir_input_line_num: # 上の注意から欠番の信号線が入力信号線であるときは、以下のようにする
            for j in range(10):  # ブリッジ故障の種類は0~9の10種類なので,10回繰り返す
                with open("bb", 'a') as f:
                    f.write(str(br_missing_line[i] - delite_count - 1) + '\n')
                br_compare_model_pass_fail.pop((br_missing_line[i] - delite_count - 1)*10)  # ブリッジ故障が発生しない信号線番号を削除
            delite_count += 1  # 削除した数をカウント
        else: # 上の注意から欠番の信号線が入力信号線以外の信号線であるときは、br_compare_model_pass_failには出力信号線は含まれていないので以下のようにする
            for j in range(10): 
                br_compare_model_pass_fail.pop((br_missing_line[i] - delite_count - cir_output_line_num - 1)*10)
                with open("bb", 'a') as f:
                    f.write(str(br_missing_line[i] - delite_count - cir_output_line_num - 1) + '\n')
            delite_count += 1  # 削除した数をカウント
            print(br_missing_line[i] - delite_count - cir_output_line_num - 1)
    
    # print("br_compare_model_pass_failの長さ：", len(br_compare_model_pass_fail))  # ブリッジ故障のパスフェイル情報の長さを表示

    # print("delite_count：",i)  # 削除した数を表示
    with open("b", 'w') as f:
        for i in range(len(br_compare_model_pass_fail)):
            f.write(str(br_compare_model_pass_fail[i]) + '\n')
    

    # # ランダムに選んだ故障のパスフェイル情報（pass_failリスト）とcompare_model_pass_failを比較して、縮退故障候補を取得する
    # st_fault_candidate, st_fault_type = get_fault_candidate(st_pass_fail, st_compare_model_pass_fail, st_fault_all_line, signal_num, br_missing_line, 0)  # 縮退故障の候補を取得(縮退故障の時、br_missing_lineは使わない、引数に設定しているから一応おいている)
    # br_fault_candidate, br_fault_type = get_fault_candidate(br_pass_fail, br_compare_model_pass_fail, br_fault_all_line, signal_num, br_missing_line, 1)  # ブリッジ故障の候補を取得

    
    # # 故障候補の中に、実際に縮退故障として選んだ信号線と故障の種類が一致するものがあるかどうかを確認
    # check_fault_candidate(st_fault_candidate, st_fault_type, st_fault_target_line, st_fault_target_type, 0)  # 縮退故障の候補を確認
    # check_fault_candidate(br_fault_candidate, br_fault_type, br_fault_target_line, br_fault_target_type, 1) # ブリッジ故障の候補を確認

